In [0]:
%sql
CREATE CATALOG IF NOT EXISTS global_weather_catalog;

In [0]:
%sql
USE CATALOG global_weather_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
%sql
USE CATALOG global_weather_catalog;
USE SCHEMA bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS global_weather_catalog.bronze.src_weather
USING PARQUET
LOCATION 's3://global-weather-pipeline/bronze/2026/'
OPTIONS (
  mergeSchema = "true",
  recursiveFileLookup = "true"
);

In [0]:
%sql
SELECT 'weather' AS stream, COUNT(*) AS rows 
FROM global_weather_catalog.bronze.src_weather;

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit, col
import datetime
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_weather")

BUCKET     = "s3://global-weather-pipeline"
CATALOG    = "global_weather_catalog"
batch_date = datetime.date.today().strftime("%Y-%m-%d")

def run_bronze_ingestion():
    logger.info("=" * 60)
    logger.info(f"Starting Bronze ingestion — stream : weather")
    logger.info(f"Batch date   : {batch_date}")
    logger.info(f"Source       : {BUCKET}/raw/ (raw CSV)")
    logger.info(f"Target table : {CATALOG}.bronze.bronze_weather")
    logger.info("=" * 60)

    try:
        # ── Check source has data ──────────────────────────
        source_count = spark.read \
            .option("header", "true") \
            .csv(f"{BUCKET}/raw/") \
            .count()

        if source_count == 0:
            logger.warning("No records in source. Bronze ingestion skipped.")
            return

        logger.info(f"[weather] Records detected in source : {source_count}")

        # ── Load raw CSV ───────────────────────────────────
        df = spark.read \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .csv(f"{BUCKET}/raw/")
        logger.info("[weather] Source data loaded")

        # ── Fix special character column names ─────────────
        df = (
            df
            .withColumnRenamed("air_quality_PM2.5",           "air_quality_pm2_5")
            .withColumnRenamed("air_quality_PM10",             "air_quality_pm10")
            .withColumnRenamed("air_quality_Carbon_Monoxide",  "air_quality_carbon_monoxide")
            .withColumnRenamed("air_quality_Ozone",            "air_quality_ozone")
            .withColumnRenamed("air_quality_Nitrogen_dioxide", "air_quality_nitrogen_dioxide")
            .withColumnRenamed("air_quality_Sulphur_dioxide",  "air_quality_sulphur_dioxide")
            .withColumnRenamed("air_quality_us-epa-index",     "air_quality_us_epa_index")
            .withColumnRenamed("air_quality_gb-defra-index",   "air_quality_gb_defra_index")
        )
        logger.info("[weather] Special character column names fixed")

        # ── Normalize all column names to lowercase ────────
        df = df.toDF(*[c.lower() for c in df.columns])
        logger.info("[weather] Column names normalized to lowercase")

        # ── Add Bronze audit columns ───────────────────────
        df_bronze = (
            df
            .withColumn("_batch_date",   lit(batch_date))
            .withColumn("_ingestion_ts", current_timestamp())
            .withColumn("_source_path",  lit(f"{BUCKET}/raw/"))
            .withColumn("_stream_name",  lit("weather"))
        )
        logger.info(f"[weather] Audit columns added — total columns : {len(df_bronze.columns)}")

        # ── Write to Bronze Delta (OVERWRITE) ─────────────
        bronze_path = f"{BUCKET}/bronze_delta/"
        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("_batch_date") \
            .save(bronze_path)
        logger.info(f"[weather] Delta overwrite complete → {bronze_path}")

        # ── Register in Unity Catalog ──────────────────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {CATALOG}.bronze.bronze_weather
            USING DELTA
            LOCATION '{bronze_path}'
        """)
        logger.info(f"[weather] Table registered : {CATALOG}.bronze.bronze_weather")

        # ── Optimize ───────────────────────────────────────
        spark.sql(f"OPTIMIZE {CATALOG}.bronze.bronze_weather ZORDER BY (country)")
        logger.info("[weather] OPTIMIZE complete")

        # ── Verify row count ───────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {CATALOG}.bronze.bronze_weather"
        ).collect()[0]["cnt"]
        logger.info(f"[weather] Rows in Bronze table after overwrite : {written}")

        # ── Null check on air quality columns ─────────────
        null_pm25 = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {CATALOG}.bronze.bronze_weather
            WHERE air_quality_pm2_5 IS NULL
        """).collect()[0]["cnt"]

        if null_pm25 > 0:
            logger.warning(f"[weather] NULL air_quality_pm2_5 found : {null_pm25}")
        else:
            logger.info("[weather] NULL check passed — 0 null air_quality_pm2_5")

        logger.info("[weather] Bronze ingestion SUCCESSFUL ✓")

    except Exception as e:
        logger.error("[weather] Bronze ingestion FAILED")
        logger.error(str(e))
        raise

    finally:
        logger.info("[weather] Bronze pipeline execution finished")
        logger.info("=" * 60)

run_bronze_ingestion()

In [0]:
%sql
select * from global_weather_catalog.bronze.bronze_weather;

In [0]:
from pyspark.sql.functions import rand, when, lit, col

logger.info("=" * 60)
logger.info("Injecting nulls, duplicates & invalid values into Bronze")
logger.info("=" * 60)

# ── Read clean Bronze ──────────────────────────────────────
df_clean = spark.table(f"{CATALOG}.bronze.bronze_weather")

# ── Inject NULLs ──────────────────────────────────────────
df_dirty = df_clean \
    .withColumn("temperature_celsius",
        when(rand() < 0.03, None).otherwise(col("temperature_celsius"))) \
    .withColumn("humidity",
        when(rand() < 0.02, None).otherwise(col("humidity"))) \
    .withColumn("wind_kph",
        when(rand() < 0.02, None).otherwise(col("wind_kph"))) \
    .withColumn("condition_text",
        when(rand() < 0.01, None).otherwise(col("condition_text"))) \
    .withColumn("air_quality_pm2_5",
        when(rand() < 0.05, None).otherwise(col("air_quality_pm2_5")))

# ── Inject invalid/outlier values ─────────────────────────
df_dirty = df_dirty \
    .withColumn("humidity",
        when(rand() < 0.005, lit(150)).otherwise(col("humidity"))) \
    .withColumn("temperature_celsius",
        when(rand() < 0.005, lit(999.0)).otherwise(col("temperature_celsius"))) \
    .withColumn("uv_index",
        when(rand() < 0.005, lit(-5.0)).otherwise(col("uv_index")))

# ── Inject duplicates (~500 rows) ─────────────────────────
df_dupes  = df_clean.sample(fraction=0.004, seed=42)
df_dirty  = df_dirty.union(df_dupes)

# ── Overwrite Bronze with dirty data ──────────────────────
bronze_path = f"{BUCKET}/bronze_delta/"
df_dirty.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("_batch_date") \
    .save(bronze_path)

# ── Verify ────────────────────────────────────────────────
from pyspark.sql.functions import sum as spark_sum
total = df_dirty.count()
null_check = df_dirty.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in ["temperature_celsius", "humidity", "wind_kph",
              "condition_text", "air_quality_pm2_5"]
]).collect()[0].asDict()

print(f"✅ Dirty Bronze written successfully!")
print(f"   Total rows (with duplicates) : {total}")
print(f"   Duplicates injected          : {total - 127646}")
print(f"\n📊 Nulls injected:")
for k, v in null_check.items():
    print(f"   {k}: {v} nulls")
print(f"\n📊 Invalid values injected:")
print(f"   humidity > 100    : {df_dirty.filter(col('humidity') > 100).count()} rows")
print(f"   temperature = 999 : {df_dirty.filter(col('temperature_celsius') == 999).count()} rows")
print(f"   uv_index < 0      : {df_dirty.filter(col('uv_index') < 0).count()} rows")